# Ropedia Academy — B4 · Implicit representations & SDF

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ChaoYue0307/ropedia-academy/blob/main/notebooks/B4.ipynb)

> **Trains an MLP to represent a signed-distance field, then extracts normals as ∇f via autograd — the implicit-geometry idea that NeRF and TSDF reuse.**
>
> 训练一个 MLP 来表示符号距离场，再用自动求导把法向作为 ∇f 提取——NeRF 与 TSDF 都复用的隐式几何思想。

This is the lesson's core example — **self-contained and runnable end to end**. It builds toy tensors, performs the lesson's key computation, and prints a real result, so you learn the concept by executing it.

Colab's default runtime already includes `torch`, `numpy`, and `networkx`, so just press **Run all** — every cell should go green. Sizes are shrunk to run on CPU; swap in a real batch and the same code scales up.

🔗 Full lesson (with the interactive demo & key terms): https://chaoyue0307.github.io/ropedia-academy/lesson/B4

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F

# Geometry as a FUNCTION f(x): signed distance to the surface; surface = {f = 0}.
sphere_sdf = lambda p, r=1.0: p.norm(dim=-1) - r   # <0 inside, >0 outside

# An implicit field: train a small MLP to BE the SDF (continuous, resolution-free).
mlp = nn.Sequential(nn.Linear(3,64), nn.Softplus(), nn.Linear(64,64), nn.Softplus(), nn.Linear(64,1))
opt = torch.optim.Adam(mlp.parameters(), 1e-3)
for _ in range(400):
    x = torch.randn(512, 3) * 1.5
    loss = ((mlp(x).squeeze(-1) - sphere_sdf(x)) ** 2).mean()
    opt.zero_grad(); loss.backward(); opt.step()
print("MLP learned the SDF, residual:", round(loss.item(), 4))

# Normals come FREE as the gradient of the field: n = ∇f, defined everywhere.
x = torch.randn(5, 3, requires_grad=True)
g = torch.autograd.grad(mlp(x).sum(), x)[0]
print("surface normals from ∇f:", tuple(F.normalize(g, dim=-1).shape))
print("same 'distance to nearest surface' -> NeRF density (B5) and TSDF fusion (D3)")

### Where to go next

- Swap the toy tensors for a real batch and watch the shapes flow through.
- Open the matching lesson for the math and an interactive figure: https://chaoyue0307.github.io/ropedia-academy/lesson/B4
- Browse every notebook: https://github.com/ChaoYue0307/ropedia-academy/tree/main/notebooks